# MuSeg-AI Thigh Segmentation â€” Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Runs [fabianbalsiger/museg-ai](https://github.com/fabianbalsiger/museg-ai) (nnU-Net `thigh-model3`)
on fat-fraction stacks **without Docker**, using nnU-Net directly with a GPU runtime.


Warning: Before uploading to Colab,  you need to run this once locally (Docker is already running on your machine) to extract the weights:


**This notebook is designed to run on Google Colab with a GPU runtime.**

## One-time setup: extract weights from Docker (run locally, requires Docker)

```bash
docker pull fabianbalsiger/museg:thigh-model3
docker create --name museg_extract fabianbalsiger/museg:thigh-model3
docker cp museg_extract:/nnUNet_trained_models ./nnUNet_trained_models
docker rm museg_extract
```

Then upload the `nnUNet_trained_models/` folder to your Google Drive at:
`MyDrive/museg_weights/nnUNet_trained_models/`

## Steps in this notebook
1. Select **Runtime â†’ Change runtime type â†’ T4 GPU**
2. Run *Install dependencies*
3. Run *Mount Google Drive* and authorise
4. Set `DRIVE_ROOT` in the *Config* cell if your Drive layout differs
5. Run all remaining cells

Data layout on Drive (same structure as local):
```
MyDrive/
  myosegmenTUM/<subject>/ImageData/<subject>_FATFRACTION/<subject>_FATFRACTION_stack*.nii
  nnUNet_trained_models/                 <- extracted from Docker above
  museg_thigh_segs/                      <- outputs written here
```

In [ ]:
import sys
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('Running in Colab:', IN_COLAB)

In [ ]:
# Install dependencies
!pip install -q "git+https://github.com/fabianbalsiger/museg-ai.git"
!pip install -q nnunet==1.7.1

# Python 3.12 removed distutils from stdlib; patch it back for nnunet
import sys
try:
    import distutils  # noqa: F401
except ModuleNotFoundError:
    try:
        import setuptools._distutils as _dt
        sys.modules['distutils'] = _dt
        for _sub in ['dir_util', 'file_util', 'errors', 'version', 'command']:
            try:
                sys.modules[f'distutils.{_sub}'] = getattr(_dt, _sub)
            except AttributeError:
                pass
        print('distutils patched for Python 3.12+')
    except Exception as _e:
        print(f'Could not patch distutils: {_e}')


In [ ]:
# Mount Google Drive
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

In [ ]:
import glob
import os
import numpy as np
from musegai import api

DRIVE_ROOT       = '/content/drive/MyDrive'
DATA_DIR         = os.path.join(DRIVE_ROOT, 'myosegmenTUM')
WEIGHTS_DIR      = os.path.join(DRIVE_ROOT, 'nnUNet_trained_models')
OUTPUT_DIR       = os.path.join(DRIVE_ROOT, 'museg_thigh_segs')

IMAGE_GLOB = os.path.join(DATA_DIR, '*', 'ImageData',
                          '*FATFRACTION', '*FATFRACTION_stack*.nii')

LABEL_MAP = {
    1:  'Vastus_Lateralis',
    2:  'Vastus_Intermedius',
    3:  'Vastus_Medialis',
    4:  'Rectus_Femoris',
    5:  'Sartorius',
    6:  'Gracilis',
    7:  'Semimembranosus',
    8:  'Semitendinosus',
    9:  'Biceps_Femoris',
    10: 'Biceps_Femoris_Short',
    11: 'Adductor_Magnus',
    12: 'Adductor_Longus',
    13: 'Adductor_Brevis',
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Data dir exists:',    os.path.isdir(DATA_DIR))
print('Weights dir exists:', os.path.isdir(WEIGHTS_DIR))
image_files = sorted(glob.glob(IMAGE_GLOB))
print(f'Found {len(image_files)} fat-fraction stacks')

In [ ]:
# Download the custom nnU-Net trainer from GitHub and register it
import urllib.request, importlib, subprocess, sys
from pathlib import Path

trainer_url = ('https://raw.githubusercontent.com/fabianbalsiger/museg-ai'
               '/main/docker/nnUNetTrainerV2_MUSEGAI.py')
nnunet_trainers = Path(importlib.util.find_spec('nnunet').origin).parent / 'training' / 'network_training'
trainer_dest    = nnunet_trainers / 'nnUNetTrainerV2_MUSEGAI.py'

if not trainer_dest.exists():
    urllib.request.urlretrieve(trainer_url, trainer_dest)
    print('Trainer downloaded to', trainer_dest)
else:
    print('Trainer already present')

# Set nnU-Net environment variables
os.environ['RESULTS_FOLDER']       = WEIGHTS_DIR
os.environ['nnUNet_results']        = WEIGHTS_DIR
os.environ['nnUNet_raw']            = '/tmp/nnunet_raw'
os.environ['nnUNet_preprocessed']   = '/tmp/nnunet_preprocessed'
print('nnU-Net env vars set')

In [ ]:
# Monkey-patch: mock docker, fix torch.load, fix scipy map_coordinates, use nnU-Net Python API
import os, sys, torch, numpy as np, scipy.ndimage as _snd
from unittest.mock import MagicMock
import musegai.api as api_module

# Block all Docker calls in musegai
api_module.docker = MagicMock()

# PyTorch 2.6+ changed torch.load default to weights_only=True;
# nnU-Net v1 checkpoints contain numpy objects blocked by that default.
_orig_torch_load = torch.load
def _patched_torch_load(f, *args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_torch_load(f, *args, **kwargs)
torch.load = _patched_torch_load

# scipy 1.14+ dropped support for several dtypes in map_coordinates.
# nnU-Net v1 can pass float16/float32 arrays that newer scipy rejects.
# Patch the binding inside the nnunet preprocessing module directly so
# the fix is inherited by forked multiprocessing workers.
_orig_map_coords = _snd.map_coordinates
def _safe_map_coordinates(input, coordinates, **kwargs):
    input = np.asarray(input, dtype=np.float64)
    return _orig_map_coords(input, coordinates, **kwargs)
_snd.map_coordinates = _safe_map_coordinates
from nnunet.preprocessing import preprocessing as _nnunet_pp
_nnunet_pp.map_coordinates = _safe_map_coordinates

def _run_model_native(model, indir, outdir):
    print('Running nnU-Net inference (no Docker)')
    model_folder = os.path.join(
        WEIGHTS_DIR, 'nnUNet', '3d_fullres', 'Task503_MuscleThigh',
        'nnUNetTrainerV2_MUSEGAI__nnUNetPlansv2.1'
    )
    print(f'  Model folder: {model_folder}')
    print(f'  Exists: {os.path.isdir(model_folder)}')
    from nnunet.inference.predict import predict_from_folder
    predict_from_folder(
        model=model_folder,
        input_folder=str(indir),
        output_folder=str(outdir),
        folds=None,
        save_npz=False,
        num_threads_preprocessing=2,
        num_threads_nifti_save=2,
        lowres_segmentations=None,
        part_id=0,
        num_parts=1,
        tta=False,
        mixed_precision=False,
        overwrite_existing=True,
        mode='normal',
        overwrite_all_in_gpu=True,
        step_size=0.5,
        checkpoint_name='model_final_checkpoint',
        segmentation_export_kwargs=None,
        disable_postprocessing=False,
    )

api_module._run_model = _run_model_native
print('Patches applied: docker mock, torch.load weights_only, scipy map_coordinates dtype cast')


In [ ]:
for nii_path in image_files:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f'{stem}_museg.nii.gz')

    if os.path.exists(out_path):
        print(f'Skipping (done): {stem}')
        continue

    print(f'\nProcessing: {stem}')
    vol = api.Volume.load(nii_path)
    print(f'  Shape: {vol.shape}  Spacing: {vol.spacing}')

    results, labels = api.segment_volumes(
        {stem: [vol, vol]},
        model='thigh-model3',
        side='left+right',
    )

    segmentation = results[stem]
    segmentation.save(out_path)
    print(f'  Saved -> {out_path}')

    seg_arr = segmentation.array
    print(f'  Labels present: {sorted(np.unique(seg_arr).tolist())}')
    print(f'  {"Label":<6} {"Muscle":<25} {"Voxels":>10}')
    print(f'  {"-"*45}')
    for idx, name in LABEL_MAP.items():
        n = int((seg_arr == idx).sum())
        if n > 0:
            print(f'  {idx:<6} {name:<25} {n:>10,}')

print('\nAll done.')